In [1]:
%load_ext autoreload
%autoreload 2

from IPython.display import display
from src.utils import build_data_from_suffix
from src.utils import save_str_ud_deprel_mismatches

In [2]:
DATA = build_data_from_suffix("syntax", csv_dir="aligned")

In [3]:
out = save_str_ud_deprel_mismatches(DATA, "new", "разъяснит", "parataxis", "appos")

In [13]:
import pandas as pd
matrix = pd.read_csv("matrix_full.csv")
for i in range(10):
    out = save_str_ud_deprel_mismatches(DATA, "full", matrix["deprel_g_str"][i], matrix["deprel_g_ud"][i], matrix["deprel_p_ud"][i], index=i+1)

# New:

### предик <-> nsubj:pass <-> nsubj

Это “ложный пассив на подлежащем” в предложениях с двумя предикациями (обычно координация/присоединение): в UD-графе корневым считается активный предикат (например, `опустил`), а пассивная предикация оформлена как `conj` (`разочарован`) с `aux:pass`; при этом конвертер, видимо, определяет `nsubj` vs `nsubj:pass` по наличию признака страдательности где-то “в окрестности” (в самом токене/его вершине/цепочке), и из-за этого подлежащее, которое синтаксически привязано к активному `root`, ошибочно получает `nsubj:pass` — пассивный признак относится не к главной вершине (head подлежащего), а к соседней пассивной клаузе, поэтому метка подлежащего переносится “не туда”.

### опред <-> acl <-> amod

Это группа ошибок “ложное `acl` вместо `amod`” при конвертации СинТагРус `опред`: конвертер решает `amod` vs `acl` по факту наличия у модификатора зависимых, и если находит хоть какие-то, переводит в `acl` (“распространенная группа/причастный оборот”), хотя в реальных примерах зависимые часто оказываются структурными (`conj` в рядах однородных определений или `parataxis` во вставках/уточнениях) и **не делают** определение распространенным в смысле UD-определения `acl`; в результате простые атрибуты, которые по правилу должны оставаться `amod`, систематически помечаются как `acl` только из‑за “нерасширяющих” зависимостей.

### аппоз <-> appos <-> flat:name

Это группа ошибок, где модель систематически предсказывает `flat:name`, а в UD-gold стоит `appos`, потому что конструкции вида “титул/родовой термин + имя” (например, “мосье Годар”, “пик Маттерхорн”) иногда ошибочно интерпретируются как многословное собственное имя; но по UD-определению это скорее аппозиция: второе имя (PROPN) непосредственно уточняет/называет первый именной узел (“мосье”, “пик”), поэтому корректная связь — `appos`, а `flat:name` должна применяться к собственным именам, состоящим из нескольких токенов внутри самой name-группы, а не к сочетанию “класс + имя”.

### разъяснит <-> appos <-> parataxis

Это группа ошибок, где модель переоценивает `parataxis` (вероятно, из‑за сильной ассоциации двоеточия/тире и “вставных кусков” с прямой речью и дискурсивным присоединением), тогда как в этих примерах после `:`/`—` идёт не самостоятельная клауза, а именной перечень/уточнение, то есть номинал конкретизирует номинал (“зрелище: пальто…”, “экипаж…: тракторист…”, “продукты: бутылки…”); поэтому по UD‑определению корректнее `appos`, а `parataxis` был бы уместен скорее при присоединении целого предложения/реплики, а не при номинальной расшифровке.

### атриб <-> nmod <-> obl 

Это группа ошибок, где в UD-gold стоит `nmod`, а модель предсказывает `obl`, потому что в данных встречаются конструкции, где именная группа (часто с предлогом: “для экономики”, “на один глаз”, “по отдельности”, “по определению”, “для них”) зависит не от существительного, а от предикативного/квази‑предикативного узла (прилагательного, глагола или даже детерминатива/местоимения) и фактически играет роль обстоятельственного/неядерного аргумента; по UD v2 такие зависимые должны размечаться как `obl`, тогда как `nmod` ограничен модификацией именных вершин, поэтому расхождение `атриб` → `nmod` (gold) ↔ `obl` (pred) можно интерпретировать как след старой практики/конвертации, где `nmod` ещё применяли при ADJ/VERB/ADV, и модель “исправляет” это в сторону современной нормы.

# Old:

### 1-компл <-> obl <-> obj 

Значительная часть выявленных расхождений между разметкой СинТагРус и UD связана с системной ошибкой конвертации объектов при глаголах, способных управлять винительным падежом. В ряде случаев зависимые в родительном падеже (в частности, при отрицании: не нашли партнёров, не нашёл камня) были автоматически размечены как obl на основании формального критерия падежа, хотя синтаксически они представляют собой прямые аргументы глагола. Согласно логике UD, определяющим является не конкретная морфологическая форма в данном контексте, а валентность глагола и статус аргумента как core argument: если глагол способен управлять объектом в винительном падеже, то соответствующий зависимый должен аннотироваться как obj, даже если в конкретном предложении реализован генитив (например, под влиянием отрицания). Следовательно, систематическая разметка таких зависимых как obl отражает не синтаксическое различие, а интерпретацию падежа как обликвного, что приводит к неконсистентности: один и тот же аргумент при одном и том же глаголе получает разные отношения (obj vs obl) в зависимости от морфологической реализации. Эти случаи следует рассматривать как ошибки конвертации (включая этап ручной проверки), а не как ошибки модели, поскольку модель в ряде примеров, предсказывая obj, фактически воспроизводит более последовательную синтаксическую интерпретацию.

### обст <-> obl <-> nmod 

Это можно описать как типичную ошибку модели на границе классов obl/nmod: она переобобщает шаблон “предложная/падежная именная группа (часто время/место) ⇒ nmod”, ориентируясь на формальные признаки (Case/наличие предлога, именная группа рядом с числом/местоимением), и игнорирует синтаксическую функцию — что в UD такие временные/локативные группы часто являются обстоятельственными зависимыми предиката и должны размечаться как obl, особенно в эллиптических/номинативных конструкциях, где вершиной может быть не глагол.




### примыкат <-> appos <-> parataxis

Это можно описать как ошибку конвертации правилами “в лоб”: конвертер сопоставляет русское отношение `примыкат` с UD-меткой `appos`, хотя в UD `appos` предназначено в первую очередь для связи **двух номиналов** (двух полноценных именных групп в аппозиции). В результате `appos` ставится слишком широко и начинает связывать не только `NOUN/PROPN`, но и предикативные/клаузы и другие не-номинальные элементы (например, глагольные/предикативные вставки в скобках или пояснения через тире), где по UD корректнее было бы `parataxis` (для “поставленных рядом” пояснительных фрагментов) или другие отношения. То есть ошибка в том, что конвертация не проверяет “номинальность” обеих частей и не различает аппозицию (N↔N) от парентетических/дискурсивных добавлений (→ `parataxis`).

### 1-компл <-> obj <-> obl

Есть выше

### сравнит <-> obl <-> advcl

Ошибка модели

### аппоз <-> appos <-> nmod

Ошибка модели

### root <-> cop <-> root

Это ошибка конвертации, связанная с неверным распознаванием конструкций с формами «быть/есть/было»: при переводе из исходной разметки конвертер интерпретирует такие предложения как копулярные и перекидывает статус предиката с «быть» на ближайшую именную/предложную группу, делая её вершиной (root), а саму форму «быть» помечая как `cop`. В результате в экзистенциальных/локативных конструкциях типа «На поле не было…», «В прошлом было…», «Есть у недвижимости…», где «быть» фактически является основным предикатом высказывания, конвертация ошибочно превращает обстоятельственный компонент (место/время/у‑группу) в “сказуемое” и получает систематическую замену `root` → `cop` для «быть».